# Train “Hey Second Brain” — custom openWakeWord model

Clean Kaggle notebook for training the wake phrase:

**`hey second brain`**

### Before running

1. Kaggle **Internet = ON**
2. Accelerator = **GPU T4 ×2**
3. Use **Save Version → Save & Run All (Commit)**

Heavy temporary files are written to Kaggle scratch storage. Only the final artifacts are copied to `/kaggle/working`.

### Final output

If training completes successfully:

`/kaggle/working/hey_secondbrain_outputs.zip`

The ZIP contains the ONNX model, YAML config, SHA256, and held-out synthetic positive clips.


In [ ]:
# Cell 0 — Kaggle workspace setup
# Heavy temporary files go to scratch storage.
# Only final outputs go to /kaggle/working.

import os
import shutil
from pathlib import Path

PERSIST_ROOT = Path("/kaggle/working")

# Find the scratch location with the most available space.
candidates = [
    Path("/kaggle/temp"),
    Path("/tmp"),
]

available = []

for p in candidates:
    try:
        p.mkdir(parents=True, exist_ok=True)
        usage = shutil.disk_usage(p)

        available.append(
            (usage.free, p)
        )

        print(
            p,
            "free:",
            f"{usage.free / 1024**3:.2f} GB"
        )

    except Exception as e:
        print(p, "unavailable:", e)


assert available, "No scratch storage available."

free_bytes, SCRATCH_BASE = max(available, key=lambda x: x[0])

assert free_bytes > 35 * 1024**3, (
    f"Not enough scratch disk. "
    f"Only {free_bytes / 1024**3:.2f} GB free."
)

WORKROOT = SCRATCH_BASE / "secondbrain"

WORKROOT.mkdir(parents=True, exist_ok=True)

os.chdir(WORKROOT)

print()
print("=" * 70)
print("SECOND BRAIN WORKSPACE")
print("=" * 70)
print("Scratch:", WORKROOT)
print("Final outputs:", PERSIST_ROOT)
print(
    "Scratch free:",
    f"{shutil.disk_usage(WORKROOT).free / 1024**3:.2f} GB"
)
print("=" * 70)

In [ ]:
# Cell 1 — Environment report. FAIL FAST if there's no GPU.
import sys, shutil, subprocess, os

print("Python:", sys.version)
print("Working dir:", os.getcwd())

if shutil.which("nvidia-smi"):
    print(
        subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
            capture_output=True,
            text=True
        ).stdout
    )
    GPU = True
else:
    GPU = False
    print("*" * 70)
    print("NO GPU DETECTED.")
    print("In Kaggle: Settings -> Accelerator -> GPU T4 x2")
    print("*" * 70)

assert GPU, "Enable GPU T4 x2 in Kaggle Settings before running."

In [ ]:
# Cell 2 — Tooling (runbook step 1).

!git clone https://github.com/dscripka/piper-sample-generator

!wget -q -O piper-sample-generator/models/en-us-libritts-high.pt \
https://github.com/rhasspy/piper-sample-generator/releases/download/v1.0.0/en-us-libritts-high.pt

!apt-get -qq -y install espeak-ng > /dev/null
!pip install -q espeak-phonemizer

!pip install -q --no-deps "openwakeword==0.6.0"

!pip uninstall -q -y onnxruntime
!pip install -q "onnxruntime-gpu==1.22.*" onnxscript scikit-learn requests scipy tqdm soundfile

!pip install -q webrtcvad || pip install -q webrtcvad-wheels
!pip install -q piper-phonemize-fix

!pip install -q \
mutagen==1.47.0 \
torchinfo==1.8.0 \
torchmetrics==1.2.0 \
speechbrain==0.5.14 \
audiomentations==0.33.0 \
torch-audiomentations==0.11.0 \
acoustics==0.2.6 \
pronouncing==0.2.0 \
deep-phonemizer==0.0.19

!pip install -q "datasets==2.21.0"


# openWakeWord base models
from importlib.metadata import version
import openwakeword.utils as oww_utils

print("openwakeword", version("openwakeword"))

assert hasattr(oww_utils, "download_models"), (
    "An old openwakeword is loaded in this session. "
    "Restart the Kaggle session and rerun."
)

oww_utils.download_models()


# Detect NVIDIA library paths dynamically
import glob as _glob
import site as _site
import os as _os

_candidates = []

for _base in _site.getsitepackages():
    _candidates.extend(
        _glob.glob(_base + "/nvidia/*/lib")
    )

_nvlibs = ":".join(sorted(_candidates))
_os.environ["NV_LIBS"] = _nvlibs

print("NV_LIBS:", _nvlibs)


# Verify ONNX Runtime can see CUDA
!export LD_LIBRARY_PATH=$NV_LIBS:$LD_LIBRARY_PATH && python -c "\
import onnxruntime as o, openwakeword, os; \
m = os.path.join(os.path.dirname(openwakeword.__file__), \
'resources', 'models', 'melspectrogram.onnx'); \
s = o.InferenceSession(m, providers=['CUDAExecutionProvider','CPUExecutionProvider']); \
print('ORT session providers:', s.get_providers())"


# Compatibility shim
from pathlib import Path

Path("oww_train_shim.py").write_text('''

import torchaudio

if not hasattr(torchaudio, "set_audio_backend"):
    torchaudio.set_audio_backend = lambda *a, **k: None

if not hasattr(torchaudio, "get_audio_backend"):
    torchaudio.get_audio_backend = lambda: "soundfile"


import scipy.special

if not hasattr(scipy.special, "sph_harm"):
    def _sph_harm(m, n, theta, phi, *args, **kwargs):
        return scipy.special.sph_harm_y(n, m, phi, theta, *args, **kwargs)

    scipy.special.sph_harm = _sph_harm


import torch

_orig_torch_load = torch.load

def _torch_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _orig_torch_load(*args, **kwargs)

torch.load = _torch_load


# torchaudio.info compatibility
if not hasattr(torchaudio, "info"):

    import soundfile as _sf

    class _AudioMetaData:
        def __init__(self, i):
            self.sample_rate = int(i.samplerate)
            self.num_frames = int(i.frames)
            self.num_channels = int(i.channels)
            self.bits_per_sample = 16
            self.encoding = "PCM_S"

    torchaudio.info = lambda p, *a, **k: _AudioMetaData(
        _sf.info(str(p))
    )


# torchaudio.load fallback
try:
    import torchcodec

except Exception:

    import soundfile as _sf2
    import torch as _torch

    def _sf_load(
        path,
        frame_offset=0,
        num_frames=-1,
        normalize=True,
        channels_first=True,
        **_kwargs
    ):

        data, sr = _sf2.read(
            str(path),
            dtype="float32",
            always_2d=True,
            start=int(frame_offset),
            frames=int(num_frames)
            if num_frames and int(num_frames) > 0
            else -1
        )

        tensor = _torch.from_numpy(
            data.T if channels_first else data
        )

        return tensor, sr

    torchaudio.load = _sf_load


import runpy

runpy.run_module(
    "openwakeword.train",
    run_name="__main__"
)

''')

!python oww_train_shim.py --help > /dev/null && echo "train.py imports OK (shim active)"


# Preflight
Path("preflight.py").write_text('''

import sys
sys.path.insert(0, "piper-sample-generator")

import torch

_o = torch.load

torch.load = lambda *a, **k: _o(
    *a,
    **{**k, "weights_only": False}
)

from espeak_phonemizer import Phonemizer

ipa = Phonemizer("en-us").phonemize(
    "hey second brain"
)

print("espeak OK:", ipa)

torch.load(
    "piper-sample-generator/models/en-us-libritts-high.pt",
    map_location="cpu"
)

print("PREFLIGHT OK: phonemizer + voice checkpoint load")

''')

!python preflight.py

print("Tooling ready.")

In [ ]:
# Cell 3 — Background/negative data + precomputed openWakeWord features.
# Heavy data lives in Kaggle scratch storage.

import os
import shutil
import subprocess
from pathlib import Path

import numpy as np
import scipy.io.wavfile
import datasets
from tqdm import tqdm

ROOT = WORKROOT

os.chdir(ROOT)


def disk_free_gb():
    return shutil.disk_usage(ROOT).free / (1024 ** 3)


def wav_count(d):
    p = Path(d)
    return len(list(p.glob("*.wav"))) if p.is_dir() else 0


print(
    f"Free scratch disk at start: "
    f"{disk_free_gb():.2f} GB"
)


# ------------------------------------------------------------
# MIT RIRs
# ------------------------------------------------------------

output_dir = ROOT / "mit_rirs"

if wav_count(output_dir) < 250:

    output_dir.mkdir(parents=True, exist_ok=True)

    rir_dataset = datasets.load_dataset(
        "davidscripka/MIT_environmental_impulse_responses",
        split="train",
        streaming=True
    )

    for row in tqdm(
        rir_dataset,
        desc="MIT RIRs"
    ):

        name = Path(
            row["audio"]["path"]
        ).name

        audio = np.asarray(
            row["audio"]["array"],
            dtype=np.float32
        )

        scipy.io.wavfile.write(
            output_dir / name,
            16000,
            (audio * 32767).astype(np.int16)
        )


# ------------------------------------------------------------
# AudioSet
# ------------------------------------------------------------

output_dir = ROOT / "audioset_16k"

N_AUDIOSET = 2000

if wav_count(output_dir) < N_AUDIOSET:

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    aset = datasets.load_dataset(
        "agkphysics/AudioSet",
        "balanced",
        split="train",
        streaming=True
    )

    aset = aset.cast_column(
        "audio",
        datasets.Audio(
            sampling_rate=16000
        )
    )

    done = 0

    for row in tqdm(
        aset,
        total=N_AUDIOSET,
        desc="AudioSet -> 16k"
    ):

        stem = str(
            row.get("video_id")
            or f"audioset_{done:05d}"
        )

        outfile = (
            output_dir /
            f"{stem}.wav"
        )

        if not outfile.exists():

            audio = np.asarray(
                row["audio"]["array"],
                dtype=np.float32
            )

            scipy.io.wavfile.write(
                outfile,
                16000,
                (audio * 32767).astype(
                    np.int16
                )
            )

        done += 1

        if done >= N_AUDIOSET:
            break


# ------------------------------------------------------------
# FMA
# ------------------------------------------------------------

output_dir = ROOT / "fma"

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

N_FMA = 360

fma_zip = ROOT / "fma_small.zip"

fma_extract = ROOT / "fma_zip"

EXPECTED_FMA_SIZE = 7679594875


if wav_count(output_dir) < 300:

    if not (
        fma_zip.exists()
        and
        fma_zip.stat().st_size
        == EXPECTED_FMA_SIZE
    ):

        subprocess.run(
            [
                "wget",
                "-c",
                "-O",
                str(fma_zip),
                "https://os.unil.cloud.switch.ch/fma/fma_small.zip"
            ],
            check=True
        )


    assert (
        fma_zip.stat().st_size
        == EXPECTED_FMA_SIZE
    ), "FMA download incomplete."


    fma_extract.mkdir(
        parents=True,
        exist_ok=True
    )

    subprocess.run(
        [
            "unzip",
            "-q",
            "-o",
            str(fma_zip),
            "fma_small/00[0-7]/*",
            "-d",
            str(fma_extract)
        ],
        check=True
    )


    mp3s = sorted(
        fma_extract.glob(
            "**/*.mp3"
        )
    )[:N_FMA]


    for mp3 in tqdm(
        mp3s,
        desc="FMA -> 16k"
    ):

        out = (
            output_dir /
            f"{mp3.stem}.wav"
        )

        if out.exists():
            continue

        subprocess.run(
            [
                "ffmpeg",
                "-hide_banner",
                "-loglevel",
                "error",
                "-y",
                "-i",
                str(mp3),
                "-ac",
                "1",
                "-ar",
                "16000",
                str(out)
            ],
            check=False
        )


# Delete large temporary FMA source files.

if fma_zip.exists():
    fma_zip.unlink()

if fma_extract.exists():
    shutil.rmtree(
        fma_extract
    )


print(
    "FMA WAVs:",
    wav_count(output_dir)
)

print(
    "Free before OWW features:",
    f"{disk_free_gb():.2f} GB"
)


# ------------------------------------------------------------
# openWakeWord precomputed features
# ------------------------------------------------------------

FEATURES = {

    "openwakeword_features_ACAV100M_2000_hrs_16bit.npy":
        17280000128,

    "validation_set_features.npy":
        184836608,
}


for fname, expected_size in FEATURES.items():

    path = ROOT / fname

    if (
        path.exists()
        and
        path.stat().st_size
        == expected_size
    ):
        print(
            fname,
            "already complete"
        )
        continue


    url = (
        "https://huggingface.co/datasets/"
        "davidscripka/openwakeword_features/"
        f"resolve/main/{fname}"
    )


    print()
    print("Downloading:", fname)
    print(
        "Free disk:",
        f"{disk_free_gb():.2f} GB"
    )


    subprocess.run(
        [
            "wget",
            "-c",
            "-O",
            str(path),
            url
        ],
        check=True
    )


    assert (
        path.stat().st_size
        == expected_size
    ), (
        f"{fname} incomplete."
    )


print()
print("=" * 70)
print("DATA READY")
print("=" * 70)

print(
    "MIT RIR:",
    wav_count(ROOT / "mit_rirs")
)

print(
    "AudioSet:",
    wav_count(ROOT / "audioset_16k")
)

print(
    "FMA:",
    wav_count(ROOT / "fma")
)

print(
    "Free scratch:",
    f"{disk_free_gb():.2f} GB"
)

print("=" * 70)

In [ ]:
# Cell 4 — Write openWakeWord config.

from pathlib import Path

yaml_path = WORKROOT / "hey_secondbrain.yaml"

yaml_path.write_text(
f"""
model_name: "hey_secondbrain"

target_phrase:
  - "hey second brain"

custom_negative_phrases:
  - "second brain"
  - "the second brain"
  - "my second brain"
  - "hey second"
  - "hey brain"
  - "second brand"
  - "second train"

n_samples: 75000
n_samples_val: 7500

tts_batch_size: 50
augmentation_batch_size: 16

augmentation_rounds: 2

piper_sample_generator_path: "{WORKROOT}/piper-sample-generator"

output_dir: "{WORKROOT}/hey_secondbrain_model"

rir_paths:
  - "{WORKROOT}/mit_rirs"

background_paths:
  - "{WORKROOT}/audioset_16k"
  - "{WORKROOT}/fma"

background_paths_duplication_rate:
  - 1
  - 1

false_positive_validation_data_path:
  "{WORKROOT}/validation_set_features.npy"

feature_data_files:
  "ACAV100M_sample":
    "{WORKROOT}/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"

batch_n_per_class:
  "ACAV100M_sample": 1024
  "adversarial_negative": 50
  "positive": 50

model_type: "dnn"

layer_size: 32

steps: 50000

max_negative_weight: 1500

target_false_positives_per_hour: 0.2
"""
)

print(yaml_path.read_text())

In [ ]:
# Cell 5 — Generate synthetic clips.

import os
import sys
import subprocess

os.chdir(WORKROOT)

env = os.environ.copy()

env["LD_LIBRARY_PATH"] = (
    env.get("NV_LIBS", "")
    + ":"
    + env.get("LD_LIBRARY_PATH", "")
)

subprocess.run(
    [
        sys.executable,
        str(WORKROOT / "oww_train_shim.py"),
        "--training_config",
        str(WORKROOT / "hey_secondbrain.yaml"),
        "--generate_clips",
    ],
    check=True,
    cwd=WORKROOT,
    env=env
)

print("Sample generation finished.")

In [ ]:
# Cell 6 — Augment clips + generate features.

import os
import sys
import glob
import subprocess
from pathlib import Path

feature_dir = (
    WORKROOT /
    "hey_secondbrain_model" /
    "hey_secondbrain"
)

feats = glob.glob(
    str(
        feature_dir /
        "*_features_*.npy"
    )
)

for f in feats:
    os.remove(f)

print(
    "cleared feature files:",
    feats
)


env = os.environ.copy()

env["LD_LIBRARY_PATH"] = (
    env.get("NV_LIBS", "")
    + ":"
    + env.get("LD_LIBRARY_PATH", "")
)


subprocess.run(
    [
        sys.executable,
        str(
            WORKROOT /
            "oww_train_shim.py"
        ),
        "--training_config",
        str(
            WORKROOT /
            "hey_secondbrain.yaml"
        ),
        "--augment_clips",
    ],
    check=True,
    cwd=WORKROOT,
    env=env
)

print("Augmentation finished.")

In [ ]:
# Cell 7 — Train wake-word model.

import os
import sys
import subprocess
from pathlib import Path

env = os.environ.copy()

env["LD_LIBRARY_PATH"] = (
    env.get("NV_LIBS", "")
    + ":"
    + env.get("LD_LIBRARY_PATH", "")
)


result = subprocess.run(
    [
        sys.executable,
        str(
            WORKROOT /
            "oww_train_shim.py"
        ),
        "--training_config",
        str(
            WORKROOT /
            "hey_secondbrain.yaml"
        ),
        "--train_model",
    ],
    cwd=WORKROOT,
    env=env
)


onnx_candidates = list(
    WORKROOT.glob(
        "**/hey_secondbrain*.onnx"
    )
)


if result.returncode != 0:

    if onnx_candidates:

        print(
            "Training process exited non-zero, "
            "but ONNX model exists."
        )

        print(
            "Continuing to packaging."
        )

    else:

        raise RuntimeError(
            "Training failed and no ONNX model was created."
        )


print(
    "ONNX candidates:",
    onnx_candidates
)

In [ ]:
# Cell 8 — Verify + package final outputs.

import hashlib
import random
import shutil
from pathlib import Path


MODEL_ROOT = (
    WORKROOT /
    "hey_secondbrain_model"
)

STAGE = (
    PERSIST_ROOT /
    "hey_secondbrain_outputs"
)


onnx_candidates = sorted(
    WORKROOT.glob(
        "**/hey_secondbrain*.onnx"
    )
)


assert onnx_candidates, (
    "No hey_secondbrain ONNX model found."
)


onnx_path = onnx_candidates[0]


if STAGE.exists():
    shutil.rmtree(STAGE)

STAGE.mkdir(
    parents=True
)


# ------------------------------------------------------------
# ONNX
# ------------------------------------------------------------

final_onnx = (
    STAGE /
    "hey_secondbrain.onnx"
)

shutil.copy2(
    onnx_path,
    final_onnx
)


sha = hashlib.sha256(
    final_onnx.read_bytes()
).hexdigest()


print(
    "Model:",
    onnx_path
)

print(
    "SHA256:",
    sha
)


# ------------------------------------------------------------
# External ONNX weights, if any
# ------------------------------------------------------------

sidecar = Path(
    str(onnx_path) + ".data"
)

if sidecar.exists():

    shutil.copy2(
        sidecar,
        STAGE /
        "hey_secondbrain.onnx.data"
    )

    print(
        "ONNX sidecar copied."
    )


# ------------------------------------------------------------
# Config
# ------------------------------------------------------------

shutil.copy2(
    WORKROOT /
    "hey_secondbrain.yaml",
    STAGE /
    "hey_secondbrain.yaml"
)


(STAGE / "SHA256.txt").write_text(
    f"{sha}  hey_secondbrain.onnx\n"
)


# ------------------------------------------------------------
# Optional TFLite
# ------------------------------------------------------------

tflites = sorted(
    WORKROOT.glob(
        "**/hey_secondbrain*.tflite"
    )
)

if tflites:

    shutil.copy2(
        tflites[0],
        STAGE /
        tflites[0].name
    )


# ------------------------------------------------------------
# Synthetic positive test samples
# ------------------------------------------------------------

test_clips = sorted(
    MODEL_ROOT.glob(
        "**/positive_test/*.wav"
    )
)


if test_clips:

    heldout = (
        STAGE /
        "heldout_synthetic_positives"
    )

    heldout.mkdir()

    selected = random.Random(0).sample(
        test_clips,
        min(
            150,
            len(test_clips)
        )
    )

    for clip in selected:

        shutil.copy2(
            clip,
            heldout /
            clip.name
        )

    print(
        "Held-out positives:",
        len(selected)
    )


# ------------------------------------------------------------
# ZIP
# ------------------------------------------------------------

zip_path = shutil.make_archive(
    str(
        PERSIST_ROOT /
        "hey_secondbrain_outputs"
    ),
    "zip",
    PERSIST_ROOT,
    "hey_secondbrain_outputs"
)


print()
print("=" * 70)
print("SECOND BRAIN MODEL COMPLETE")
print("=" * 70)

print(
    "ONNX:",
    final_onnx
)

print(
    "ZIP:",
    zip_path
)

print(
    "ZIP size:",
    f"{Path(zip_path).stat().st_size / 1024**2:.2f} MB"
)

print("=" * 70)